# UIT-VSFC Data Preprocessing Pipeline

Preprocesses the [UIT-VSFC](https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback)
(Vietnamese Students' Feedback Corpus) using the **same** cleaning pipeline as VSMEC
(`DataPreprocessing.ipynb`), then converts integer sentiment labels to 3-class strings and
exports ready-to-use train / validation / test CSVs.

**Run this notebook before the `VSFC_ensemble/` notebooks (01–07).**

Steps:
1. Install dependencies
2. Repo / path setup
3. Load UIT-VSFC from Hugging Face
4. Apply Vietnamese text cleaning (`text_preprocess.preprocess_vietnamese_text`)
5. Convert `sentiment` integers → 3-class string labels
6. Deduplicate within each split (never across splits)
7. Diagnostics — label distribution, text-length stats, before/after samples
8. Export CSVs + `label_map.json` to `data/processed/vsfc/`

> **Note:** Stopwords are intentionally not removed — BERT models rely on words like
> *không* (negation) and *rất* (very). This matches the VSMEC preprocessing decision.

## 0. Install dependencies

In [ ]:
%pip install -q datasets pandas numpy matplotlib seaborn

## 1. Repo setup & imports

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

def find_tm_research_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if p.name == 'tm_research':
            return p
        if (p / 'tm_research').is_dir():
            return p / 'tm_research'
    raise RuntimeError('Set TM_REPO_ROOT or run from repo checkout')

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = os.environ.get('TM_REPO_ROOT', '/content/drive/MyDrive/thesis/topicmodeling')
    TM_ROOT = Path(REPO_ROOT) / 'tm_research'
else:
    TM_ROOT = find_tm_research_root()
    REPO_ROOT = str(TM_ROOT.parent)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

OUTPUT_DIR = TM_ROOT / 'data' / 'processed' / 'vsfc'
RANDOM_SEED = 42

print('REPO_ROOT  :', REPO_ROOT)
print('OUTPUT_DIR :', OUTPUT_DIR)

## 2. Load UIT-VSFC dataset

`datasets` 4.x no longer runs Hub loading scripts (`vietnamese_students_feedback.py`).
Use the official Parquet export branch: `revision="refs/convert/parquet"`.

In [ ]:
from datasets import load_dataset

VSFC_HF_ID = 'uitnlp/vietnamese_students_feedback'
# datasets>=4: script-based repos fail unless you load the parquet export branch.
vsfc = load_dataset(VSFC_HF_ID, revision='refs/convert/parquet')
print(vsfc)

def _hf_split_to_df(split) -> pd.DataFrame:
    df = split.to_pandas()[['sentence', 'sentiment']].copy()
    df.rename(columns={'sentence': 'text_raw'}, inplace=True)
    return df

train_raw = _hf_split_to_df(vsfc['train'])
val_raw   = _hf_split_to_df(vsfc['validation'])
test_raw  = _hf_split_to_df(vsfc['test'])

print(f'\nRaw sizes — Train: {len(train_raw):,} | Val: {len(val_raw):,} | Test: {len(test_raw):,}')
print('\nSentiment integer distribution (train):')
print(train_raw['sentiment'].value_counts().sort_index())

## 3. Apply Vietnamese text cleaning

In [ ]:
from tm_research.text_preprocess import preprocess_vietnamese_text

print('Cleaning train split ...')
train_raw['text'] = train_raw['text_raw'].apply(preprocess_vietnamese_text)
print('Cleaning val split ...')
val_raw['text']   = val_raw['text_raw'].apply(preprocess_vietnamese_text)
print('Cleaning test split ...')
test_raw['text']  = test_raw['text_raw'].apply(preprocess_vietnamese_text)
print('Done.')

# Quick sanity: before / after samples
samples = train_raw.sample(5, random_state=RANDOM_SEED)
print('\nBEFORE → AFTER (5 samples):')
for _, row in samples.iterrows():
    print(f'  RAW : {row["text_raw"]}')
    print(f'  CLN : {row["text"]}\n')

## 4. Convert sentiment integers → 3-class string labels

In [ ]:
from tm_research.eval.vsfc_labels import (
    VSFC_INT_TO_LABEL, POLARITY_CLASSES, POLARITY_TO_IDX,
)

print('Sentiment mapping:', VSFC_INT_TO_LABEL)


def add_vsfc_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['label']    = df['sentiment'].map(VSFC_INT_TO_LABEL)
    df['label_id'] = df['label'].map(POLARITY_TO_IDX)
    return df


train_raw = add_vsfc_labels(train_raw)
val_raw   = add_vsfc_labels(val_raw)
test_raw  = add_vsfc_labels(test_raw)

# Validate: no unmapped values
for name, df in [('train', train_raw), ('val', val_raw), ('test', test_raw)]:
    assert df['label'].notna().all(), f'Unmapped sentiment values in {name}'
    assert set(df['label'].unique()) <= set(POLARITY_CLASSES), \
        f'Unexpected labels in {name}: {set(df["label"].unique()) - set(POLARITY_CLASSES)}'

print('\nLabel distribution (train):')
print(train_raw['label'].value_counts()[POLARITY_CLASSES])

## 5. Deduplication (within each split)

In [ ]:
def dedup_split(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
    removed = before - len(df)
    print(f'  {name}: {before:,} → {len(df):,}  (removed {removed} duplicates)')
    return df


print('Deduplication (within each split only — HF partition is preserved):')
train_df = dedup_split(train_raw, 'train')
val_df   = dedup_split(val_raw,   'val')
test_df  = dedup_split(test_raw,  'test')

# Shuffle train only (matches VSMEC export convention; val/test keep original order)
train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print('\nTrain split shuffled with random_state=42.')

## 6. Diagnostics

In [ ]:
# Label distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['label'].value_counts()[POLARITY_CLASSES]
    ax.bar(POLARITY_CLASSES, counts.values)
    ax.set_title(f'UIT-VSFC {name} — sentiment')
    ax.set_ylabel('Count')
    for bar, v in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Text-length before / after cleaning (train)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_raw['text_raw'].str.len(), bins=50, edgecolor='black')
axes[0].set_title('Train — raw text length')
axes[0].set_xlabel('Characters')
axes[1].hist(train_df['text'].str.len(), bins=50, edgecolor='black')
axes[1].set_title('Train — cleaned text length')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.show()

print('Text length stats (train, after cleaning):')
print(train_df['text'].str.len().describe().round(1).to_string())

## 7. Export

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_COLS = ['text_raw', 'text', 'sentiment', 'label', 'label_id']

splits = {
    'vsfc_train_final.csv': train_df,
    'vsfc_val_final.csv':   val_df,
    'vsfc_test_final.csv':  test_df,
}

for filename, df in splits.items():
    path = OUTPUT_DIR / filename
    df[EXPORT_COLS].to_csv(path, index=False)
    print(f'Saved {len(df):,} rows → {path}')

# label_map.json — built from sorted unique train labels (matches utils_io convention)
train_labels = sorted(train_df['label'].unique().tolist())
label_map = {
    'label2id': {lab: i for i, lab in enumerate(train_labels)},
    'id2label': {str(i): lab for i, lab in enumerate(train_labels)},
}
lmap_path = OUTPUT_DIR / 'label_map.json'
with open(lmap_path, 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print(f'Saved label_map → {lmap_path}')
print('label2id:', label_map['label2id'])
print('\nFinal columns:', EXPORT_COLS)